# The Impact of M&A on Inventor Mobility and Innovation
## Draft notebook for the GitHub package

This notebook is a narrative companion to the construction and analysis pipeline in the repository. It is written to do four jobs at once:

1. explain how the data are built and how the main empirical designs work,
2. present a disciplined headline story,
3. distinguish robust findings from fragile or method-dependent ones,
4. serve as a base for a sequence of public-facing writeups such as LinkedIn posts.

## Current headline story

The cleanest story is **not** that mergers and acquisitions uniformly reduce innovation output.  
The stronger and more defensible claim is narrower:

> **M&A appears to change inventor mobility and the direction of inventive search more clearly than it changes aggregate output, with especially important effects on the acquiror side.**

In particular, the most interesting pattern is that **acquiror-side inventor mobility rises after deals, and exploration becomes more prominent**. By contrast, effects on patents, citations, and related output measures are often more fragile across methods or complicated by pre-trends.

## What this notebook treats as primary versus secondary

### Primary findings
- Acquiror-side post-deal inventor mobility.
- Acquiror-side exploration outcomes.
- Heterogeneity by firm size, especially where negative baseline effects weaken among larger firms.

### Secondary findings
- Aggregate output measures such as patent counts, citations, and quality proxies.
- Arrivals and departures as supporting evidence for organizational reshuffling.
- Target-side effects, unless they are unusually stable across methods.

## Important caution

This draft reflects the current state of the project and the review notes. Some statements below are intentionally cautious because a few results are sensitive to estimator choice, event-study shape, or pre-trend concerns.

## Repository orientation

A clean GitHub version of the project should separate the work into:

- **construction scripts**: build cleaned firm-year and inventor-year / inventor-event panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,
- **notebooks**: one main narrative notebook, plus optional shorter diagnostics notebooks.

This notebook is meant to be the **main interpretive notebook** rather than the place where every heavy computation happens.

## Empirical design in plain language

The project uses publicly listed firms and patent-linked inventors to study what happens around merger and acquisition events.

At a high level, there are two complementary designs:

### 1. Firm-year panel
This asks whether treated firms differ from comparable untreated firms after a deal. It is useful for aggregate outcomes such as:
- patent counts,
- citations,
- novelty and exploration measures,
- inventor flows in and out of the firm.

### 2. Inventor-year / inventor-event panel
This asks whether inventors attached to treated firms behave differently after a deal relative to matched controls. This is especially useful for:
- mobility,
- exploration,
- inventor-level productivity,
- within-firm heterogeneity.

The value of the project is that it combines both levels:
the firm panel captures organizational outcomes, while the inventor panel helps interpret the mechanism.

## Identification strategy

The main estimators are:
- baseline fixed-effects DiD,
- event studies,
- Sun–Abraham interaction-weighted event studies,
- Borusyak–Jaravel–Spiess imputation,
- selected matching and stacked-panel designs.

The reason for using multiple estimators is straightforward: staggered treatment timing can make simple two-way fixed-effects event studies misleading, especially when treatment effects evolve over time or differ across cohorts.

That is why, in interpretation, this notebook gives more weight to:
- patterns that survive across multiple estimators,
- results without severe pre-trend problems,
- economically sensible magnitudes and shapes.

It gives less weight to:
- isolated significant coefficients,
- event studies with visibly unstable pre-trends,
- very large or implausible coefficients,
- patterns that reverse sign across methods without a clear explanation.

In [ ]:
# Core imports for the notebook
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Repository root: adjust after cloning if needed
REPO_ROOT = Path(".").resolve()

# Example output folders used by the analysis script
MODEL_OUT = REPO_ROOT / "Model_outputs"
PLOTS_OUT = MODEL_OUT / "Plots"

print("Repository root:", REPO_ROOT)
print("Model outputs:", MODEL_OUT)

## Step 1. Load the key outputs

The notebook is easiest to use if the heavy analysis has already been run and the output CSVs / plots already exist.

The helper file added to the repository can build:
- summary statistics tables,
- panel diagnostics,
- robustness scorecards,
- compact headline tables for the notebook.

The next cell assumes those helper functions are available.

In [ ]:
# These imports assume the helper file from this draft has been added to the repo.
# If the file is placed elsewhere, adjust the import path.
try:
    from notebook_additions_minimal import (
        build_headline_scorecard,
        summarize_firm_panel,
        summarize_stacked_panel,
        summarize_inventor_event_panel,
        build_result_inventory,
        write_notebook_support_outputs,
    )
    HELPERS_AVAILABLE = True
except Exception as e:
    HELPERS_AVAILABLE = False
    print("Helper import failed:", e)

## Step 2. Recommended summary statistics to include

A public-facing notebook should show basic descriptive structure before jumping into causal estimates.

I recommend including four compact tables:

1. **Firm panel summary**  
   Number of firms, years, treated firms, and baseline means for headline outcomes.

2. **Stacked firm-panel summary**  
   Number of treated and control firm-event units, balance over the event window, and treated share.

3. **Inventor-event panel summary**  
   Number of inventors, inventor-event units, treated and control units, and event-window balance.

4. **Headline baseline means**  
   Means at `t = -1` for the outcomes that appear in the narrative.

This helps readers anchor effect sizes and see the scale of the analysis.

In [ ]:
# Example usage once the relevant dataframes are loaded in memory:
#
# firm_summary = summarize_firm_panel(firm_panel, outcomes=["xi_real", "arriving_inventors_count", "exploration_firm"])
# acq_stack_summary = summarize_stacked_panel(stacked_acquiror, label="Acquiror stack")
# tgt_stack_summary = summarize_stacked_panel(stacked_target, label="Target stack")
# inv_summary = summarize_inventor_event_panel(inv_es, label="Inventor event panel")
#
# display(firm_summary)
# display(acq_stack_summary)
# display(tgt_stack_summary)
# display(inv_summary)

## Step 3. How to think about the results

### A. What is a true headline result?
A result is headline-worthy if most of the following are true:
- the sign is stable across reasonable specifications,
- event-study pre-trends are not obviously problematic,
- the magnitude is economically interpretable,
- the estimate aligns with the project's mechanism story,
- advanced estimators do not overturn it.

### B. What is a supporting result?
A result is useful but secondary if:
- it supports the mechanism,
- it is intuitive,
- but it is weaker or less stable than the main finding.

### C. What should stay out of the headline?
Results should stay out of the headline if:
- they are driven by a single estimator,
- they rely on implausibly large coefficients,
- they have visibly non-flat pre-trends,
- or the interpretation depends on too many caveats.

## Proposed interpretation of the current evidence

### 1. The strongest story is on the acquiror side
The review notes suggest the cleanest results are concentrated on the acquiror side rather than the target side.

This is useful because it also makes economic sense: integration frictions, organizational redesign, and shifts in technological search are more likely to show up where post-merger integration is actively managed.

### 2. Mobility and exploration are more credible than broad output decline
The clearest evidence appears to concern:
- inventor movement,
- exploration behavior,
- and composition changes in inventive activity.

That is a better story than a blanket claim that M&A simply reduces innovation.

### 3. Firm size seems to moderate effects
The heterogeneity notes suggest that some negative baseline effects weaken or disappear among larger firms. That is an important moderating result and fits a plausible narrative:
larger firms may be better able to absorb integration costs, preserve research capacity, or reallocate inventors effectively.

### 4. Exploitation should not be a centerpiece
For presentation, it is sensible to focus on **exploration** and treat exploitation more cautiously. In many settings it is mechanically linked to exploration, so emphasizing both equally can make the story look redundant.

## Draft headline paragraph for the notebook

**Draft version**

This project studies how mergers and acquisitions affect firms' inventive activity and the behavior of individual inventors. Across firm-level and inventor-level panels, the most persuasive pattern is not a uniform decline in innovation output. Instead, the data point more clearly to a reorganization of inventive effort after deals, especially on the acquiror side. Post-merger periods are associated with changes in inventor mobility and stronger exploration-oriented behavior, while broad output measures such as patents and citations are less stable across estimators and often complicated by pre-trend concerns. The evidence also suggests that firm size moderates these effects, with larger firms appearing better able to absorb or offset negative baseline impacts.

## Results section structure for the final notebook

I would organize the final notebook results section exactly like this:

### Section 1. Main descriptive setup
- sample definition,
- event window,
- matching logic,
- panel sizes.

### Section 2. Headline result: acquiror inventor mobility
- baseline DiD,
- event study,
- one advanced estimator figure,
- short interpretation.

### Section 3. Headline result: exploration
- baseline DiD,
- event study,
- advanced estimator comparison,
- explain why this is more informative than exploitation.

### Section 4. Supporting result: inventor inflows / outflows
- use only if the signs are stable enough,
- keep brief.

### Section 5. Heterogeneity by firm size
- show one clean specification,
- interpret as a moderator rather than a second headline.

### Section 6. What does *not* survive cleanly
- patent counts,
- citation-based quality outcomes,
- any result with major pre-trend problems.

## Robustness philosophy

A good public notebook does not try to make every statistically significant coefficient look equally important.

Instead, it should say:
- what survived,
- what weakened,
- what looks fragile,
- and why that matters.

That makes the project look more credible, not less.

In [ ]:
# Example: build a compact result inventory from exported significance tables
#
# inventory = build_result_inventory(MODEL_OUT)
# display(inventory.head(20))
#
# The inventory is useful for quickly checking:
# - which outcomes appear significant,
# - in which role,
# - under which family of estimators,
# - and whether they belong in the headline story.

## Suggested headline scorecard

The notebook should include one compact scorecard with rows like:

- acquiror inventor mobility,
- acquiror exploration,
- arriving inventors,
- departing inventors,
- patents,
- citations,
- innovation quality.

And columns like:

- baseline DiD,
- event study shape,
- Sun–Abraham,
- BJS,
- overall judgment.

The purpose is not to automate interpretation perfectly.  
It is to force a disciplined summary.

In [ ]:
# Example scorecard construction
#
# scorecard = build_headline_scorecard(
#     baseline_csv=MODEL_OUT / "inventor_year_event_study" / "inventor_year_event_study_significance_selected.csv",
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# display(scorecard)

## Preliminary interpretation of the scorecard

The scorecard should be read conservatively:

- **Strong** means the result is stable, interpretable, and narratively central.
- **Moderate** means it is promising but should be presented with caveats.
- **Weak / fragile** means it may still be economically interesting, but it should not drive the public-facing claim.

## Draft “main findings” bullets for the notebook

### Main finding 1
**Acquiror-side inventor mobility rises after deals.**

Why this matters:
- mobility is directly linked to organizational adjustment after M&A,
- it is easier to interpret than many abstract output metrics,
- and it helps explain why output effects may be mixed.

### Main finding 2
**Exploration increases more clearly than broad output falls.**

Why this matters:
- it suggests a change in the direction of search rather than a simple collapse in inventive activity,
- it connects naturally to post-merger reorganization,
- and it is more distinctive than exploitation, which is partly its mirror image.

### Main finding 3
**Negative baseline effects weaken for larger firms.**

Why this matters:
- it introduces economically meaningful heterogeneity,
- and it gives the paper a richer contribution than just documenting an average effect.

## Draft “what we do not claim” section

This project does **not** currently support a simple claim that M&A universally depresses innovation.

That stronger claim would require:
- cleaner agreement across estimators,
- fewer pre-trend concerns,
- and more stable output effects across the patent and citation measures.

That does not weaken the project. It sharpens it.

## Additional statistics worth adding before the GitHub version is finalized

I recommend adding the following compact extras:

1. **Baseline means at `t = -1`** for the headline outcomes.  
   This lets readers interpret magnitudes.

2. **Counts of treated and control units** in each main stacked panel.  
   This helps readers trust the sample construction.

3. **Pre-trend diagnostic table** for the headline event studies.  
   Even a simple count of significant pre-period coefficients is useful.

4. **Outcome availability table** showing how much missingness exists by outcome.  
   This is especially useful in inventor-level panels.

5. **One robustness scorecard** combining baseline and advanced methods.  
   This is probably the single most valuable addition for presentation.

## Draft sequence for LinkedIn posts

### Post 1 — The question
How do mergers and acquisitions affect innovation inside firms? Most people expect a simple output story. The data suggest something subtler.

### Post 2 — The mechanism
The clearest post-merger changes appear in inventor mobility and the direction of search. The organization changes before the output numbers fully tell a story.

### Post 3 — Exploration
One of the strongest patterns is an increase in exploration-oriented behavior on the acquiror side. That suggests reallocation and search broadening after deals.

### Post 4 — Heterogeneity
Larger firms appear better able to absorb or offset some negative baseline effects. Integration capacity may matter as much as the deal itself.

### Post 5 — Methods
Why multiple DiD estimators matter in staggered-adoption settings, and why not every significant coefficient should make it into the headline.

## Suggested next empirical additions

If you want to strengthen the notebook further, the highest-value additions are:

1. a compact summary-statistics block,
2. a pre-trend diagnostic table,
3. a results scorecard that classifies outcomes as strong / moderate / fragile,
4. one or two polished headline figures.

I would **not** expand the scope much beyond that unless a new analysis clearly improves the story.

## Final draft conclusion

The project's strongest contribution is to show that the consequences of M&A for innovation are visible first in **people and search behavior**, not just in headline output counts. The most credible evidence points to post-deal changes in inventor mobility and exploration, especially for acquirors, while aggregate output measures are more mixed and should be interpreted cautiously. That narrower claim is both more defensible and more interesting.

In [ ]:
# Optional convenience cell:
# once the heavy analysis has been run and outputs exist, use the helper file
# to create notebook-ready support CSVs in one step.
#
# support = write_notebook_support_outputs(
#     output_root=MODEL_OUT,
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# support